In [ ]:
!pip install --upgrade ipywidgets
!jupyter nbextension enable --py widgetsnbextension
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 59.5 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: jupyterlab-widgets
    Found existing installation: jupyterlab_widgets 1.1.11
    Uninstalling jupyterlab_widgets-1.1.11:
      Successfully uninstalled jupyterlab_widgets-1.1.11
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.2
    Uninstalling ipywidgets-7.7.2:
      Successfully uninstalled ipywidgets-7.7.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autovizwidget 0.21.0 requires pandas<2.0.0,>=0.20.1, but you have pandas 2.2.3 which is incompatible.
hdijupyterutils 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pand

In [1]:
!pip uninstall streamlit_jupyter

In [3]:
!pip install -r require_dev_view_20250317.txt -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.1.1 requires nvidia-ml-py3==7.352.0, which is not installed.
autogluon-multimodal 1.1.1 requires jsonschema<4.22,>=4.18, but you have jsonschema 4.23.0 which is incompatible.
autogluon-multimodal 1.1.1 requires omegaconf<2.3.0,>=2.1.1, but you have omegaconf 2.3.0 which is incompatible.
autogluon-multimodal 1.1.1 requires scikit-learn<1.4.1,>=1.3.0, but you have scikit-learn 1.5.2 which is incompatible.
autogluon-multimodal 1.1.1 requires scipy<1.13,>=1.5.4, but you have scipy 1.14.1 which is incompatible.
autogluon-multimodal 1.1.1 requires torch<2.4,>=2.2, but you have torch 2.4.1.post100 which is incompatible.
autogluon-multimodal 1.1.1 requires transformers[sentencepiece]<4.41.0,>=4.38.0, but you have transformers 4.49.0 which is incompatible.
autogluon-timeseries 1.1.1 requires gluonts=

In [7]:
import boto3
import faiss
import numpy as np
import pandas as pd
import os
import json
import logging
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import dspy
import ipywidgets as widgets
from IPython.display import display

# Configuration Settings
BUCKET_NAME = "sagemaker-us-east-1-188494237500"
SET_R_DIRECTORY = "dev/pdf"
FAISS_INDEX_R_PATH = "dev/idx"
INDEX_DIM = 384  # Adjusted to match 'all-MiniLM-L6-v2' embedding size

# AWS Config
s3_client = boto3.client('s3')
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

# Load Embedding Model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# DSPy Configuration
pipeline = dspy.Module()

# Define DSPy Signatures
class RationaleSignature(dspy.Signature):
    query: str = dspy.InputField()
    sources: list = dspy.InputField()
    ranked_response: str = dspy.OutputField()
    rationale: str = dspy.OutputField()

class ChainOfThoughtQASignature(dspy.Signature):
    query: str = dspy.InputField()
    context: str = dspy.InputField()
    reasoning_steps: str = dspy.OutputField()
    final_answer: str = dspy.OutputField()

rationale_model = dspy.Predict(RationaleSignature)
cot_qa_model = dspy.Predict(ChainOfThoughtQASignature)

# Function to Load PDFs from S3
def load_pdfs_from_s3(bucket_name, prefix=''):
    """Loads PDFs from S3 and extracts text."""
    pdf_texts = []
    try:
        response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
        if 'Contents' not in response:
            logging.warning(f"No files found in {prefix}")
            return []

        for obj in response['Contents']:
            file_key = obj['Key']
            pdf_obj = s3_client.get_object(Bucket=bucket_name, Key=file_key)
            pdf_reader = PdfReader(pdf_obj['Body'])
            text = "\n".join([page.extract_text() for page in pdf_reader.pages if page.extract_text()])
            if text:
                pdf_texts.append(text)

    except Exception as e:
        logging.error(f"Error loading PDFs: {e}")

    return pdf_texts

# Load PDFs from Set R
##def load_all_pdfs():
    return load_pdfs_from_s3(BUCKET_NAME, prefix=SET_R_DIRECTORY)

def load_all_pdfs():
    texts = load_pdfs_from_s3(BUCKET_NAME, prefix=SET_R_DIRECTORY)
    print(f"📄 Loaded {len(texts)} documents.")
    return texts

# Vectorization & FAISS Indexing
def create_faiss_index(texts):
    """Creates a FAISS index for the provided texts."""
    if not texts:
        logging.warning("No texts provided for indexing.")
        return None, None, None

    embeddings = embedding_model.encode(texts, convert_to_numpy=True)
    index = faiss.IndexFlatL2(INDEX_DIM)
    index.add(embeddings)
    return index, embeddings, texts

# Retrieve Relevant Chunks
def retrieve_text(query, index, texts, k=3):
    """Retrieves the top-k most relevant text chunks based on FAISS similarity search."""
    if index is None or not texts:
        logging.error("Index is not initialized.")
        return []

    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, k)
    return [{"text": texts[i], "source": f"Document {i}"} for i in indices[0] if i < len(texts)]

# AWS Bedrock API Call for Text Generation
def query_bedrock_titan(query):
    """Calls AWS Bedrock Titan model directly via API."""
    try:
        response = bedrock.invoke_model(
            modelId="amazon.titan-text-v3",
            body=json.dumps({"inputText": query}),
            accept="application/json",
            contentType="application/json"
        )
        response_body = json.loads(response["body"].read().decode("utf-8"))
        return response_body.get("outputText", "No response from model.")

    except Exception as e:
        logging.error(f"Error querying AWS Bedrock Titan: {e}")
        return "Error generating response."

# Interactive Widgets
load_button = widgets.Button(description="Load Set R PDFs")
query_input = widgets.Text(placeholder="Ask a question...")
category_dropdown = widgets.Dropdown(
    options=["Reports (Set R)", "All"],
    value="Reports (Set R)",
    description="Category:",
)
output_area = widgets.Output()

texts = []
faiss_index = None

def load_documents(_):
    global texts, faiss_index
    output_area.clear_output()
    with output_area:
        print("Loading and indexing documents...")
        texts = load_all_pdfs()
        faiss_index, _, texts = create_faiss_index(texts)
        print("✅ Documents loaded and indexed.")

load_button.on_click(load_documents)

def handle_query(_):
    output_area.clear_output()
    user_query = query_input.value
    selected_category = category_dropdown.value
    
    with output_area:
        if user_query:
            if selected_category == "Reports (Set R)":
                retrieved_chunks = retrieve_text(user_query, faiss_index, texts, k=3)
                sources = [chunk["text"] for chunk in retrieved_chunks]

                if sources:
                    ai_response = query_bedrock_titan(user_query)
                    print(f"\n**AI Response:** {ai_response}\n")
                    print("📄 **Top Retrieved Sources:**")
                    for chunk in retrieved_chunks:
                        print(f"- {chunk['source']}: {chunk['text'][:300]}...")  # Truncate for display
                else:
                    print("No relevant documents found.")

query_button = widgets.Button(description="Ask AI")
query_button.on_click(handle_query)

# Display Widgets
display(load_button, query_input, category_dropdown, query_button, output_area)

Button(description='Load Set R PDFs', style=ButtonStyle())

Text(value='', placeholder='Ask a question...')

Dropdown(description='Category:', options=('Reports (Set R)', 'All'), value='Reports (Set R)')

Button(description='Ask AI', style=ButtonStyle())

Output()